# 🚀 DyGLib Enhanced - Training Modes & MRR Metric

This notebook runs complete experiments with:
- **3 Training Modes**: Full, Random 10%, Compressed 10%
- **MRR Metric**: Mean Reciprocal Rank alongside AUC and AP
- **Binary Search Window Compression**: Intelligent data reduction

## 📋 Instructions:
1. **Enable GPU** in notebook settings
2. **Change dataset/model** in Cell 2
3. **Run all cells** sequentially
4. **Wait ~2-3 hours** for complete results

## 🎯 Quick Test:
Change `num_epochs` to 5 and `num_runs` to 1 for faster testing

In [ ]:
# ========================================
# ENVIRONMENT SETUP & IMPORT
# ========================================

# Clone your enhanced DyGLib repository
!git clone https://github.com/utkarsh-mishra9/DyGLib.git
!mv DyGLib/* ./
!rm -rf DyGLib/

# Install dependencies
!pip install scikit-learn

# Verify GPU and setup
import torch
import os

print("🚀 Environment Setup:")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    GPU_DEVICE = "0"
else:
    print("Using CPU")
    GPU_DEVICE = "-1"

# Verify your enhancements
print("\n✅ Checking modifications:")
try:
    from utils.metrics import compute_mrr
    print("✅ MRR function ready")
except ImportError as e:
    print(f"❌ MRR import failed: {e}")

# Check train_mode argument exists
import subprocess
result = subprocess.run(['python', 'train_link_prediction.py', '--help'], 
                       capture_output=True, text=True)
if 'train_mode' in result.stdout:
    print("✅ Train mode argument detected")
else:
    print("❌ Train mode argument missing")

print("\n🎯 Ready for experiments!")

In [ ]:
# ========================================
# DATASET SELECTION
# ========================================

# 🎯 CHOOSE YOUR DATASET AND MODEL HERE:
DATASET_NAME = "wikipedia"  # Change this: wikipedia, reddit, mooc, lastfm, uci, etc.
MODEL_NAME = "TGN"          # Change this: TGN, DyGFormer, TGAT, GraphMixer

print(f"📊 Selected Dataset: {DATASET_NAME}")
print(f"🤖 Selected Model: {MODEL_NAME}")
print(f"🔥 GPU Device: {GPU_DEVICE}")

# Available datasets:
print("\n📋 Available Datasets:")
print("  - wikipedia (157k interactions, fast)")
print("  - reddit (672k interactions, medium)")
print("  - mooc (411k interactions, medium)")
print("  - lastfm (1.3M interactions, large)")
print("  - uci, Flights, CanParl, USLegis, etc.")

print("\n🤖 Available Models:")
print("  - TGN (recommended, fast)")
print("  - DyGFormer (advanced, slower)")
print("  - TGAT (classic, medium speed)")
print("  - GraphMixer, CAWN, etc.")

In [ ]:
# ========================================
# DATASET DOWNLOAD & PREPROCESSING
# ========================================

print(f"📥 Downloading {DATASET_NAME} dataset...")
!python download_dataset.py {DATASET_NAME} --preprocess

# Verify dataset is ready
import os
processed_path = f"processed_data/{DATASET_NAME}/ml_{DATASET_NAME}.csv"
if os.path.exists(processed_path):
    print(f"\n✅ Dataset {DATASET_NAME} ready for training!")
    
    # Show dataset info
    import pandas as pd
    df = pd.read_csv(processed_path)
    print(f"📈 Dataset size: {len(df):,} interactions")
    print(f"📁 Processed files location: processed_data/{DATASET_NAME}/")
else:
    print(f"❌ Dataset {DATASET_NAME} not found. Check preprocessing.")
    
# List available preprocessed files
!ls -lh processed_data/{DATASET_NAME}/

In [ ]:
# ========================================
# TRAINING: FULL DATASET (BASELINE)
# ========================================

print("🎯 Training with FULL dataset...")
print("=" * 50)
print("This uses the complete 70% training split as baseline.")
print("Expected training time: ~45-60 minutes on GPU\n")

!python train_link_prediction.py \
    --dataset_name {DATASET_NAME} \
    --model_name {MODEL_NAME} \
    --train_mode full \
    --num_epochs 50 \
    --num_runs 5 \
    --gpu {GPU_DEVICE}

print("\n✅ Full dataset training completed!")
print(f"📁 Logs saved to: logs/{MODEL_NAME}/{DATASET_NAME}/")

In [ ]:
# ========================================
# TRAINING: RANDOM 10% SAMPLE
# ========================================

print("🎯 Training with RANDOM 10% sample...")
print("=" * 50)
print("This randomly samples 10% of the training data.")
print("Expected training time: ~8-10 minutes on GPU\n")

!python train_link_prediction.py \
    --dataset_name {DATASET_NAME} \
    --model_name {MODEL_NAME} \
    --train_mode random_10 \
    --num_epochs 50 \
    --num_runs 5 \
    --gpu {GPU_DEVICE}

print("\n✅ Random 10% training completed!")

In [ ]:
# ========================================
# TRAINING: COMPRESSED 10% (YOUR METHOD)
# ========================================

print("🎯 Training with COMPRESSED 10% (Binary Search Window)...")
print("=" * 50)
print("This intelligently compresses training data using binary search window compression.")
print("Expected training time: ~8-10 minutes on GPU")
print("Expected improvement: Better performance than random 10%\n")

!python train_link_prediction.py \
    --dataset_name {DATASET_NAME} \
    --model_name {MODEL_NAME} \
    --train_mode compressed_10 \
    --num_epochs 50 \
    --num_runs 5 \
    --gpu {GPU_DEVICE}

print("\n✅ Compressed 10% training completed!")
print("🔍 Look for 'Binary Search Window Compress' output above to see compression details.")

In [ ]:
# ========================================
# EVALUATION: ALL THREE MODES
# ========================================

print("📊 Evaluating all three trained models...")
print("=" * 60)
print("Running comprehensive evaluation with random negative sampling.")
print("This provides detailed performance analysis.\n")

modes = ['full', 'random_10', 'compressed_10']
mode_names = ['Full Dataset', 'Random 10%', 'Compressed 10%']

for mode, name in zip(modes, mode_names):
    print(f"\n🔍 Evaluating {name}:")
    print("-" * 40)
    
    !python evaluate_link_prediction.py \
        --dataset_name {DATASET_NAME} \
        --model_name {MODEL_NAME} \
        --train_mode {mode} \
        --negative_sample_strategy random \
        --num_runs 5 \
        --gpu {GPU_DEVICE}

print("\n✅ All evaluations completed!")
print("📊 Results analysis in next cell...")

In [ ]:
# ========================================
# RESULTS ANALYSIS & COMPARISON
# ========================================

print("📊 FINAL RESULTS COMPARISON")
print("=" * 60)

# Extract final results from logs
print(f"🎯 Results for {MODEL_NAME} on {DATASET_NAME}:\n")

# Show compression effectiveness first
print("🔍 Compression Details:")
!grep "Binary Search Window Compress" logs/{MODEL_NAME}/{DATASET_NAME}/*/*.log | head -1

print("\n📈 INDIVIDUAL RUN RESULTS:")
print("=" * 40)

# Extract individual results for each mode
print("\n🎯 Test Results (Individual Runs):")
!grep "test.*\[" logs/{MODEL_NAME}/{DATASET_NAME}/*/*.log | grep -E "average_precision|roc_auc|mrr"

print("\n📊 AVERAGE RESULTS WITH STANDARD DEVIATION:")
print("=" * 50)

# Extract averages
!grep "average test" logs/{MODEL_NAME}/{DATASET_NAME}/*/*.log | tail -9

print("\n" + "=" * 60)
print("📋 RESULTS SUMMARY")
print("=" * 60)

# Manual extraction for clear comparison
import re
import glob

try:
    log_files = glob.glob(f"logs/{MODEL_NAME}/{DATASET_NAME}/*/*.log")
    
    results_summary = {}
    
    for log_file in log_files:
        with open(log_file, 'r') as f:
            content = f.read()
            
        # Extract training size to identify mode
        train_size_match = re.search(r'training dataset has (\d+) interactions', content)
        
        if train_size_match:
            train_size = int(train_size_match.group(1))
            
            # Identify mode based on training size and compression marker
            if train_size > 50000:  # Likely full dataset
                mode = 'Full Dataset'
            elif 'Binary Search Window Compress' in content:
                mode = 'Compressed 10%'
            else:
                mode = 'Random 10%'
                
            # Extract final averages
            ap_match = re.search(r'average test average_precision, ([\d\.]+) ± ([\d\.]+)', content)
            auc_match = re.search(r'average test roc_auc, ([\d\.]+) ± ([\d\.]+)', content)
            mrr_match = re.search(r'average test mrr, ([\d\.]+) ± ([\d\.]+)', content)
            
            if ap_match and auc_match and mrr_match:
                results_summary[mode] = {
                    'ap': (float(ap_match.group(1)), float(ap_match.group(2))),
                    'auc': (float(auc_match.group(1)), float(auc_match.group(2))),
                    'mrr': (float(mrr_match.group(1)), float(mrr_match.group(2))),
                    'train_size': train_size
                }
    
    # Display clean comparison
    if results_summary:
        print(f"\n📈 PERFORMANCE COMPARISON ({DATASET_NAME} - {MODEL_NAME}):")
        print("=" * 70)
        
        for mode, results in results_summary.items():
            print(f"\n🎯 {mode}:")
            print(f"   Training Size:     {results['train_size']:,} interactions")
            print(f"   Average Precision: {results['ap'][0]:.4f} ± {results['ap'][1]:.4f}")
            print(f"   ROC AUC:           {results['auc'][0]:.4f} ± {results['auc'][1]:.4f}")
            print(f"   MRR:               {results['mrr'][0]:.4f} ± {results['mrr'][1]:.4f}")
        
        # Calculate improvements
        if 'Compressed 10%' in results_summary and 'Random 10%' in results_summary:
            comp_ap = results_summary['Compressed 10%']['ap'][0]
            rand_ap = results_summary['Random 10%']['ap'][0]
            comp_mrr = results_summary['Compressed 10%']['mrr'][0]
            rand_mrr = results_summary['Random 10%']['mrr'][0]
            
            ap_improvement = ((comp_ap - rand_ap) / rand_ap) * 100
            mrr_improvement = ((comp_mrr - rand_mrr) / rand_mrr) * 100
            
            print(f"\n🔥 KEY FINDINGS:")
            print("=" * 30)
            print(f"📊 Compressed 10% vs Random 10%:")
            print(f"   AP Improvement:  {ap_improvement:+.2f}%")
            print(f"   MRR Improvement: {mrr_improvement:+.2f}%")
            
            if 'Full Dataset' in results_summary:
                full_ap = results_summary['Full Dataset']['ap'][0]
                full_mrr = results_summary['Full Dataset']['mrr'][0]
                ap_retention = (comp_ap / full_ap) * 100
                mrr_retention = (comp_mrr / full_mrr) * 100
                
                print(f"\n💎 Compressed 10% vs Full Dataset:")
                print(f"   AP Retention:    {ap_retention:.1f}%")
                print(f"   MRR Retention:   {mrr_retention:.1f}%")
                print(f"   Data Reduction:  90% (using only 10% of training data)")
    
except Exception as e:
    print(f"Could not parse results automatically. Error: {e}")
    print("\nShowing raw results:")
    !grep -A1 -B1 "average test" logs/{MODEL_NAME}/{DATASET_NAME}/*/*.log

print(f"\n✅ Experiment completed for {DATASET_NAME} with {MODEL_NAME}!")
print(f"📁 All results saved in: logs/{MODEL_NAME}/{DATASET_NAME}/")
print(f"💾 Models saved in: saved_models/{MODEL_NAME}/{DATASET_NAME}/")
print(f"📄 JSON results in: saved_results/{MODEL_NAME}/{DATASET_NAME}/")

In [ ]:
# ========================================
# DOWNLOAD RESULTS & FILES
# ========================================

print("📦 Preparing results for download...")

# Create a summary results file
import datetime

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
summary_file = f"experiment_summary_{DATASET_NAME}_{MODEL_NAME}_{timestamp}.txt"

with open(summary_file, 'w') as f:
    f.write(f"DyGLib Enhanced Experiment Summary\n")
    f.write(f"=================\n\n")
    f.write(f"Dataset: {DATASET_NAME}\n")
    f.write(f"Model: {MODEL_NAME}\n")
    f.write(f"Training Modes: Full, Random 10%, Compressed 10%\n")
    f.write(f"Epochs: 50\n")
    f.write(f"Runs: 5\n")
    f.write(f"GPU: {'Yes' if GPU_DEVICE != '-1' else 'No'}\n")
    f.write(f"Timestamp: {datetime.datetime.now()}\n\n")

# Add results to summary
import subprocess
result = subprocess.run(['grep', '-r', 'average test', f'logs/{MODEL_NAME}/{DATASET_NAME}/'], 
                       capture_output=True, text=True)
with open(summary_file, 'a') as f:
    f.write("Final Results:\n")
    f.write("=============\n")
    f.write(result.stdout)

print(f"✅ Summary saved to: {summary_file}")

# Create zip file with key results
zip_file = f"dyglib_results_{DATASET_NAME}_{MODEL_NAME}_{timestamp}.zip"
!zip -r {zip_file} logs/{MODEL_NAME}/{DATASET_NAME}/ {summary_file} saved_results/{MODEL_NAME}/{DATASET_NAME}/ 2>/dev/null || echo "Zip created with available files"

print(f"📦 Results packaged in: {zip_file}")
print("\n💡 To download:")
print(f"   - Right-click on '{zip_file}' in file browser and download")
print(f"   - Or use: files.download('{zip_file}') in a code cell")

# List what's included
print("\n📋 Package contents:")
!unzip -l {zip_file} 2>/dev/null || echo "Contents: logs, results, summary"

# Display final file sizes
print("\n📁 File sizes:")
!ls -lh {zip_file} {summary_file}

# 🎉 Experiment Complete!

## 📊 What You Got:
- **3 Training Modes** compared: Full, Random 10%, Compressed 10%
- **3 Metrics** reported: Average Precision, ROC-AUC, **MRR**
- **Individual run results** and statistical summaries
- **Performance improvements** of compression over random sampling

## 📈 Key Insights:
- **Compressed 10%** should outperform **Random 10%** with same data size
- **Binary search window compression** preserves temporal patterns
- **MRR metric** provides ranking quality assessment
- **90% data reduction** with minimal performance loss

## 📝 Next Steps:
1. **Download results** using the zip file above
2. **Try different datasets** by changing `DATASET_NAME` in Cell 2
3. **Test other models** by changing `MODEL_NAME`
4. **Analyze compression effectiveness** across different data types

## 🔬 For Research:
- Compare compression ratios across datasets
- Analyze temporal pattern preservation
- Study computational efficiency gains
- Extend to other dynamic graph tasks

**Repository**: https://github.com/utkarsh-mishra9/DyGLib  
**Enhanced with**: Training modes, MRR metric, binary search compression